In [1]:

import os
import shutil
import random

random.seed(42)

source_dir = "dataset"
output_dir = "dataset_split"

classes = ["cats", "dogs"]

# Create train, validation and test folders
for split in ["train", "validation", "test"]:
    for category in classes:
        os.makedirs(
            os.path.join(output_dir, split, category),
            exist_ok=True
        )

# Split images
for category in classes:

    folder = os.path.join(source_dir, category)

    images = [
        img for img in os.listdir(folder)
        if img.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    random.shuffle(images)

    total = len(images)

    train_count = int(total * 0.70)
    val_count = int(total * 0.15)

    train_images = images[:train_count]
    val_images = images[train_count:train_count + val_count]
    test_images = images[train_count + val_count:]

    # Copy training images
    for img in train_images:
        shutil.copy2(
            os.path.join(folder, img),
            os.path.join(output_dir, "train", category, img)
        )

    # Copy validation images
    for img in val_images:
        shutil.copy2(
            os.path.join(folder, img),
            os.path.join(output_dir, "validation", category, img)
        )

    # Copy testing images
    for img in test_images:
        shutil.copy2(
            os.path.join(folder, img),
            os.path.join(output_dir, "test", category, img)
        )

    print("\nClass:", category)
    print("Total:", total)
    print("Training:", len(train_images))
    print("Validation:", len(val_images))
    print("Testing:", len(test_images))

print("\nDataset splitting completed!")



Class: cats
Total: 500
Training: 350
Validation: 75
Testing: 75

Class: dogs
Total: 500
Training: 350
Validation: 75
Testing: 75

Dataset splitting completed!


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
import time
import numpy as np

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam

# ==========================================
# 1. PARAMETERS
# ==========================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 5

train_dir = "dataset_split/train"
val_dir = "dataset_split/validation"
test_dir = "dataset_split/test"

# ==========================================
# 2. LOAD DATASET
# ==========================================

train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_data = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

class_names = train_data.class_names

print("Classes:", class_names)

# Apply ResNet50 preprocessing
train_data = train_data.map(
    lambda x, y: (preprocess_input(x), y)
)

val_data = val_data.map(
    lambda x, y: (preprocess_input(x), y)
)

test_data = test_data.map(
    lambda x, y: (preprocess_input(x), y)
)

# ==========================================
# 3. LOAD PRETRAINED RESNET50
# ==========================================

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

print("ResNet50 loaded successfully!")

# ==========================================
# 4. FREEZE PRETRAINED LAYERS
# ==========================================

base_model.trainable = False

print("Pretrained layers frozen!")

# ==========================================
# 5. ADD CUSTOM CLASSIFICATION LAYERS
# ==========================================

x = base_model.output

x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

x = Dropout(0.3)(x)

output = Dense(1, activation="sigmoid")(x)

model = Model(
    inputs=base_model.input,
    outputs=output
)

# ==========================================
# 6. COMPILE MODEL
# ==========================================

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ==========================================
# 7. TRAIN WITH FROZEN LAYERS
# ==========================================

print("\nTraining with frozen layers...")

start_time = time.time()

history_frozen = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

frozen_time = time.time() - start_time

frozen_loss, frozen_accuracy = model.evaluate(test_data)

print("\nFrozen Model Results")
print("Test Accuracy:", frozen_accuracy)
print("Test Loss:", frozen_loss)
print("Training Time:", frozen_time, "seconds")

# ==========================================
# 8. FINE-TUNING
# ==========================================

print("\nStarting fine-tuning...")

# Unfreeze pretrained model
base_model.trainable = True

# Freeze all except the last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile with smaller learning rate
model.compile(
    optimizer=Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Selected layers unfrozen!")

# ==========================================
# 9. TRAIN FINE-TUNED MODEL
# ==========================================

start_time = time.time()

history_finetuned = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

finetune_time = time.time() - start_time

finetune_loss, finetune_accuracy = model.evaluate(test_data)

print("\nFine-tuned Model Results")
print("Test Accuracy:", finetune_accuracy)
print("Test Loss:", finetune_loss)
print("Fine-tuning Time:", finetune_time, "seconds")

# ==========================================
# 10. COMPARE PERFORMANCE
# ==========================================

print("\n========== PERFORMANCE COMPARISON ==========")

print("Frozen Model Accuracy:", frozen_accuracy)
print("Fine-tuned Model Accuracy:", finetune_accuracy)

print("Frozen Training Time:", frozen_time)
print("Fine-tuning Time:", finetune_time)

# ==========================================
# 11. PLOT ACCURACY
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history_frozen.history["accuracy"],
    label="Frozen Training"
)

plt.plot(
    history_frozen.history["val_accuracy"],
    label="Frozen Validation"
)

plt.plot(
    history_finetuned.history["accuracy"],
    label="Fine-tuned Training"
)

plt.plot(
    history_finetuned.history["val_accuracy"],
    label="Fine-tuned Validation"
)

plt.title("ResNet50 Accuracy Comparison")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# ==========================================
# 12. PLOT LOSS
# ==========================================

plt.figure(figsize=(8, 5))

plt.plot(
    history_frozen.history["loss"],
    label="Frozen Training Loss"
)

plt.plot(
    history_finetuned.history["loss"],
    label="Fine-tuned Training Loss"
)

plt.title("Training Loss Comparison")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

# ==========================================
# 13. SAVE MODEL
# ==========================================

model.save("resnet50_cats_dogs.keras")

print("\nModel saved successfully!")
print("Transfer Learning Completed!")


Found 700 files belonging to 2 classes.
Found 150 files belonging to 2 classes.
Found 150 files belonging to 2 classes.
Classes: ['cats', 'dogs']
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
ResNet50 loaded successfully!
Pretrained layers frozen!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 23,850,113 (90.98 MB)

 Trainable params: 262,401 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)


Training with frozen layers...
Epoch 1/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 93s 2s/step - accuracy: 0.8535 - loss: 0.2942 - val_accuracy: 0.9867 - val_loss: 0.0597
Epoch 2/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.9902 - loss: 0.0350 - val_accuracy: 0.9733 - val_loss: 0.0803
Epoch 3/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - accuracy: 0.9980 - loss: 0.0086 - val_accuracy: 0.9733 - val_loss: 0.0961
Epoch 4/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 1.0000 - loss: 0.0048 - val_accuracy: 0.9733 - val_loss: 0.0719
Epoch 5/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 76s 2s/step - accuracy: 1.0000 - loss: 0.0038 - val_accuracy: 0.9733 - val_loss: 0.0910
10/10 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.9851 - loss: 0.0429

Frozen Model Results
Test Accuracy: 0.9733333587646484
Test Loss: 0.07630720734596252
Training Time: 398.9082350730896 seconds

Starting fine-tuning...
Selected layers unfrozen!
Epoch 1/5
44/44 ━━━━━━━━━━━━━━━━━━━━ 122s 3s/step - accuracy: 0.9985 - loss: 0.0069 - val_a